# 03. 계단 주행 rosbag 녹화

## 이 Notebook으로 하는 일

현재 ROS graph의 map, localization, AprilTag, navigation command, compressed camera, LiDAR, odometry, TF를 하나의 bag에 기록합니다. 다른 Notebook이나 mission recorder와 독립된 **수동 기록 전용** 도구입니다.

## 셀 실행 방법

회색 code cell은 **Python 코드**입니다. 일반 terminal에 붙여 넣지 말고 Jupyter에서 `Shift+Enter`로 한 셀씩 실행하십시오. 각 셀은 앞 셀에서 만든 `CONFIG`, `ROS_ENV`, artifact 경로와 recorder process를 사용하므로 중간 셀부터 실행하지 않습니다. Kernel을 닫으면 recorder 소유권을 잃으므로 반드시 정상 종료 셀을 먼저 실행합니다.

## 녹화 mode 선택과 실행 순서

이 Notebook은 process를 시작하지 않고 현재 graph만 기록합니다. 먼저 필요한 mode를 선택합니다.

- **센서만 기록:** sensor stack을 실행한 뒤 `bash notebooks/check_environment.sh recording`을 통과합니다.
- **평면 navigation 기록:** [`docs/ROSBAG-RECORD`](../docs/ROSBAG-RECORD)의 sensor, navigation, robot tunnel/supervisor terminal을 실행한 뒤 `navigation`과 `recording` 검사를 모두 통과합니다.
- **AprilTag 포함 기록:** 위 과정에 detector terminal을 추가하고 `/tag_detections` message도 확인합니다.

그 다음 첫 code cell의 `RUN_LABEL`을 실제 층·방향·작업명으로 바꾸고 위에서 아래로 실행합니다. Preflight가 통과하면 녹화 시작 셀을 한 번만 실행하고, `.bag.active`가 표시된 뒤에만 외부 이동 작업을 시작합니다. 주행이 끝나면 로봇을 먼저 정지시키고 정상 종료 셀, 최종 검증 셀 순서로 실행합니다.

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import shutil
import signal
import subprocess
import time
import yaml

WORKSPACE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_PATH = WORKSPACE / "config.env"
ROS_SETUP = Path("/opt/ros/noetic/setup.bash")
DEVEL_SETUP = WORKSPACE / "devel/setup.bash"

if not CONFIG_PATH.is_file() or not ROS_SETUP.is_file() or not DEVEL_SETUP.is_file():
    raise RuntimeError("workspace, config.env, or ROS Noetic setup is missing")

def read_config(path):
    values = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line and not line.startswith("#"):
            key, value = line.split("=", 1)
            values[key] = value
    return values

def sourced_environment():
    command = f"source {ROS_SETUP} && source {DEVEL_SETUP} && env -0"
    completed = subprocess.run(["bash", "-lc", command], check=True, stdout=subprocess.PIPE)
    environment = os.environ.copy()
    for entry in completed.stdout.split(b"\0"):
        if entry and b"=" in entry:
            key, value = entry.split(b"=", 1)
            environment[key.decode()] = value.decode()
    return environment

CONFIG = read_config(CONFIG_PATH)
ROS_ENV = sourced_environment()
ROS_ENV["ROS_MASTER_URI"] = f"http://{CONFIG['ROS_MASTER_HOST']}:{CONFIG['ROS_MASTER_PORT']}"
route = subprocess.run(["ip", "-4", "route", "get", CONFIG["ROS_MASTER_HOST"]], check=True, text=True, stdout=subprocess.PIPE).stdout.split()
ROS_ENV["ROS_IP"] = route[route.index("src") + 1]
ROS_ENV.pop("ROS_HOSTNAME", None)

OUTPUT_DIR = WORKSPACE / "stair_captures"
RUN_LABEL = "stair_3F_to_4F_UP"  # 실제 층, 방향, 작업명으로 변경
BAG_STEM = f"{RUN_LABEL}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
ARTIFACT = OUTPUT_DIR / f"{BAG_STEM}.bag"
RECORD_PROCESS = None
print(f"artifact={ARTIFACT}")

In [ ]:
POINTCLOUD_TOPIC = CONFIG["POINTCLOUD_TOPIC"]
CAMERA_IMAGE_TOPIC = CONFIG["CAMERA_IMAGE_TOPIC"].rstrip("/")
CAMERA_COMPRESSED_TOPIC = f"{CAMERA_IMAGE_TOPIC}/compressed"
CAMERA_INFO_TOPIC = CONFIG["CAMERA_INFO_TOPIC"]
TAG_TOPIC = CONFIG["TAG_DETECTIONS_TOPIC"]
print(f"camera={CAMERA_COMPRESSED_TOPIC}")
print(f"lidar={POINTCLOUD_TOPIC}")
print(f"tags={TAG_TOPIC}")

## 기록 topic

위 설정 셀은 `config.env`의 실제 camera, LiDAR, tag topic을 사용합니다. 영상은 detector가 소비하는 compressed stream으로 변환합니다. 필요 없는 `-a` 전체 기록 대신 재현에 필요한 topic을 명시해 용량과 분석 범위를 통제합니다. `TOPICS`는 기록 시도 목록이고 `MANDATORY_TOPICS`는 최종 bag에 반드시 message가 있어야 하는 sensor evidence입니다.

In [ ]:
TOPICS = [
    "/map", "/amcl_pose", "/particlecloud", TAG_TOPIC,
    "/initialpose", "/move_base_simple/goal", "/move_base/status",
    "/move_base/feedback", "/move_base/result", "/navigation/cmd_vel",
    "/stair/cmd_vel", CAMERA_COMPRESSED_TOPIC, CAMERA_INFO_TOPIC,
    POINTCLOUD_TOPIC, "/livox/imu", "/tron/wheel_odom_raw",
    "/scan", "/tf", "/tf_static",
]
MANDATORY_TOPICS = {
    CAMERA_COMPRESSED_TOPIC, CAMERA_INFO_TOPIC, POINTCLOUD_TOPIC,
    "/tron/wheel_odom_raw", "/scan", "/tf", "/tf_static",
}
print("topics:", *TOPICS, sep="\n- ")

## 사전 검사

필수 topic의 type, 다른 rosbag recorder 유무, 파일 충돌, 디스크 여유 공간을 확인합니다. 10 GiB 미만이면 고해상도 영상과 LiDAR 기록을 시작하지 않습니다.

In [ ]:
expected_types = {
    CAMERA_COMPRESSED_TOPIC: "sensor_msgs/CompressedImage",
    CAMERA_INFO_TOPIC: "sensor_msgs/CameraInfo",
    POINTCLOUD_TOPIC: "livox_ros_driver2/CustomMsg",
    "/tron/wheel_odom_raw": "nav_msgs/Odometry",
    "/scan": "sensor_msgs/LaserScan",
    "/tf": "tf2_msgs/TFMessage",
    "/tf_static": "tf2_msgs/TFMessage",
}
for topic, expected in expected_types.items():
    completed = subprocess.run(["rostopic", "type", topic], env=ROS_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, timeout=3)
    actual = completed.stdout.strip()
    if actual != expected:
        raise RuntimeError(f"topic unavailable or wrong type: {topic} expected={expected} actual={actual}")
recorder_pattern = "[r]osbag record|/rosbag/[r]ecord"
other_recorders = subprocess.run(["pgrep", "-af", recorder_pattern], text=True, stdout=subprocess.PIPE).stdout.strip()
if other_recorders:
    raise RuntimeError(f"another rosbag recorder is already running:\n{other_recorders}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
active_path = ARTIFACT.with_suffix(ARTIFACT.suffix + ".active")
if ARTIFACT.exists() or active_path.exists():
    raise RuntimeError(f"artifact path already exists: {ARTIFACT}")
free_gib = shutil.disk_usage(OUTPUT_DIR).free / (1024 ** 3)
if free_gib < 10.0:
    raise RuntimeError(f"at least 10 GiB free space is required; free={free_gib:.2f} GiB")
print(f"preflight OK; free={free_gib:.2f} GiB")

## 녹화 시작

이 셀을 한 번 실행합니다. `.bag.active`가 생성된 뒤에만 녹화가 준비된 것으로 판단합니다. 생성되지 않거나 process가 조기 종료되면 즉시 실패합니다.

In [ ]:
if RECORD_PROCESS is not None and RECORD_PROCESS.poll() is None:
    raise RuntimeError("this kernel already owns a running recorder")
command = ["rosbag", "record", "--buffsize=1024", "--chunksize=768", "-O", str(ARTIFACT), *TOPICS]
RECORD_PROCESS = subprocess.Popen(command, env=ROS_ENV, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, start_new_session=True)
deadline = time.monotonic() + 10.0
while time.monotonic() < deadline and not active_path.is_file():
    if RECORD_PROCESS.poll() is not None:
        raise RuntimeError(f"rosbag exited before readiness: code={RECORD_PROCESS.returncode}")
    time.sleep(0.2)
if not active_path.is_file():
    RECORD_PROCESS.send_signal(signal.SIGINT)
    RECORD_PROCESS.wait(timeout=10)
    raise RuntimeError("rosbag did not create .bag.active within 10 seconds")
print(f"recording started: pid={RECORD_PROCESS.pid} active={active_path}")

## 녹화 상태

주행 중 필요할 때 반복 실행합니다. process가 살아 있고 `.active` 파일 크기가 증가해야 정상입니다.

In [ ]:
if RECORD_PROCESS is None or RECORD_PROCESS.poll() is not None:
    raise RuntimeError("recorder is not running")
size_mib = active_path.stat().st_size / (1024 ** 2)
print(f"recording pid={RECORD_PROCESS.pid} size={size_mib:.1f} MiB path={active_path}")

## 정상 종료와 finalize

외부 이동을 먼저 정지시킨 뒤 실행합니다. rosbag에 SIGINT를 보내 index를 기록하고 `.bag.active`를 최종 `.bag`으로 finalize하도록 최대 15초 기다립니다. 성공 기준은 `recording finalized:`와 `.bag` 경로 출력입니다. Kernel을 먼저 종료하지 마십시오.

In [ ]:
if RECORD_PROCESS is None:
    raise RuntimeError("this kernel does not own a recorder")
if RECORD_PROCESS.poll() is None:
    RECORD_PROCESS.send_signal(signal.SIGINT)
    try:
        RECORD_PROCESS.wait(timeout=15)
    except subprocess.TimeoutExpired:
        RECORD_PROCESS.kill()
        RECORD_PROCESS.wait(timeout=5)
        raise RuntimeError("rosbag did not finalize after SIGINT")
if RECORD_PROCESS.returncode != 0:
    raise RuntimeError(f"rosbag exited with code {RECORD_PROCESS.returncode}")
if active_path.exists() or not ARTIFACT.is_file():
    raise RuntimeError(f"bag finalization failed: active={active_path.exists()} final={ARTIFACT.exists()}")
print(f"recording finalized: {ARTIFACT}")

## 최종 bag 검증과 복구

파일 존재만으로 성공 처리하지 않습니다. `rosbag info --yaml`이 읽을 수 있고 duration이 양수이며 모든 필수 topic에 실제 message가 있어야 합니다. 성공하면 duration과 topic별 message 수가 출력됩니다.

Kernel이 종료돼 recorder 소유권을 잃었다면 새 recorder를 시작하지 마십시오. terminal에서 `pgrep -af '/rosbag/record|rosbag.*record'`로 기존 process를 확인하고 그 process를 시작한 원래 terminal/kernel에서 정상 종료합니다. `.bag.active`만 남은 경우 원본을 보존한 채 `rosbag reindex <file>.bag.active` 복구 여부를 별도로 판단하며, 이 Notebook은 손상 파일을 자동 수정하거나 삭제하지 않습니다.

In [ ]:
completed = subprocess.run(["rosbag", "info", "--yaml", str(ARTIFACT)], env=ROS_ENV, check=True, text=True, stdout=subprocess.PIPE, timeout=60)
info = yaml.safe_load(completed.stdout)
if not isinstance(info, dict) or float(info.get("duration", 0.0)) <= 0.0:
    raise RuntimeError("bag has no positive duration")
message_counts = {entry["topic"]: int(entry["messages"]) for entry in info.get("topics", [])}
missing = sorted(topic for topic in MANDATORY_TOPICS if message_counts.get(topic, 0) <= 0)
if missing:
    raise RuntimeError(f"mandatory topics have no messages: {missing}")
print(f"validated duration={info['duration']:.2f}s size={ARTIFACT.stat().st_size / (1024 ** 3):.2f} GiB")
for topic in sorted(message_counts):
    print(f"{topic}: {message_counts[topic]}")